# 📊 Consumer Trend Signal Predictor
### AI-Powered Market Intelligence from Product Reviews

**Run order:**
1. Cell 1 — Install dependencies
2. Cell 2 — Mount Drive + write source files
3. Cell 3 — Choose: Train fresh OR load saved model
4. Cell 4 — Launch dashboard

> First run takes ~10 mins. Every run after = under 1 min (loads from Drive)

---
## ⚙️ Cell 1 — Install Dependencies

In [1]:
!pip install bertopic sentence-transformers transformers torch \
             plotly wordcloud tqdm rich openai streamlit pyngrok -q
print('✅ All dependencies installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 73.7 MB/s eta 0:00:00
✅ All dependencies installed!


---
## 📁 Cell 2 — Mount Drive + Write Source Files

In [2]:
from google.colab import drive
drive.mount('/drive')

import os
os.makedirs('src', exist_ok=True)
os.makedirs('/drive/MyDrive/consumer-trend-predictor/models', exist_ok=True)
print('✅ Drive mounted!')

Mounted at /drive
✅ Drive mounted!


In [4]:
%%writefile src/preprocessor.py
import re, unicodedata
from typing import Optional
import logging
logger = logging.getLogger(__name__)

def _normalize_unicode(text): return unicodedata.normalize('NFKC', text)
def _remove_urls(text): return re.sub(r'https?://\S+|www\.\S+', ' ', text)
def _remove_html(text): return re.sub(r'<[^>]+>', ' ', text)
def _remove_emails(text): return re.sub(r'\S+@\S+', ' ', text)
def _normalize_whitespace(text): return re.sub(r'\s+', ' ', text).strip()
def _remove_repeated_chars(text): return re.sub(r'(.)\1{2,}', r'\1\1', text)
def _remove_emojis(text):
    return re.compile('['
        u'\U0001F600-\U0001F64F'
        u'\U0001F300-\U0001F5FF'
        u'\U0001F680-\U0001F9FF'
        u'\U00002702-\U000027B0'
        ']+', flags=re.UNICODE).sub(' ', text)

class TextPreprocessor:
    def __init__(self, remove_urls=True, remove_html=True, remove_emails=True,
                 remove_emojis=True, lowercase=True, min_length=20):
        self.remove_urls=remove_urls; self.remove_html=remove_html
        self.remove_emails=remove_emails; self.remove_emojis=remove_emojis
        self.lowercase=lowercase; self.min_length=min_length

    def clean(self, text):
        if not isinstance(text, str): return None
        text = _normalize_unicode(text)
        if self.remove_html: text = _remove_html(text)
        if self.remove_urls: text = _remove_urls(text)
        if self.remove_emails: text = _remove_emails(text)
        if self.remove_emojis: text = _remove_emojis(text)
        text = _remove_repeated_chars(text)
        text = _normalize_whitespace(text)
        if self.lowercase: text = text.lower()
        return None if len(text) < self.min_length else text

    def clean_batch(self, texts):
        cleaned, valid_indices = [], []
        for i, text in enumerate(texts):
            result = self.clean(text)
            if result is not None:
                cleaned.append(result); valid_indices.append(i)
        return cleaned, valid_indices

def preprocess(texts, **kwargs):
    return TextPreprocessor(**kwargs).clean_batch(texts)

Writing src/preprocessor.py


In [5]:
%%writefile src/topic_model.py
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
import pandas as pd
import logging
logger = logging.getLogger(__name__)

class ConsumerTopicModel:
    def __init__(self, min_topic_size=15, top_n_words=10):
        self.min_topic_size=min_topic_size; self.top_n_words=top_n_words
        self.model=None; self.topics=None; self.probs=None

    def build(self):
        self.model = BERTopic(
            embedding_model=SentenceTransformer('all-MiniLM-L6-v2'),
            umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42),
            hdbscan_model=HDBSCAN(min_cluster_size=self.min_topic_size, metric='euclidean', prediction_data=True),
            top_n_words=self.top_n_words, verbose=True)
        return self

    def fit(self, texts):
        self.topics, self.probs = self.model.fit_transform(texts)
        return self

    def get_topic_table(self):
        info = self.model.get_topic_info()
        info = info[info['Topic'] != -1]
        info['top_words'] = info['Topic'].apply(lambda t: ', '.join([w for w, _ in self.model.get_topic(t)[:5]]))
        return info[['Topic','Count','Name','top_words']].reset_index(drop=True)

    def get_topic_per_doc(self, texts):
        df = pd.DataFrame({'text': texts, 'topic_id': self.topics})
        topic_info = self.model.get_topic_info().set_index('Topic')['Name'].to_dict()
        df['topic_name'] = df['topic_id'].map(topic_info)
        return df

    def save(self, path):
        self.model.save(path, serialization='safetensors', save_ctfidf=True,
                       save_embedding_model='all-MiniLM-L6-v2')

    def load(self, path):
        self.model = BERTopic.load(path)
        return self

Writing src/topic_model.py


In [6]:
%%writefile src/sentiment.py
from transformers import pipeline
import pandas as pd
import logging
logger = logging.getLogger(__name__)

class ThemeSentimentScorer:
    def __init__(self, model_name='cardiffnlp/twitter-roberta-base-sentiment-latest'):
        self.pipe = pipeline('sentiment-analysis', model=model_name, tokenizer=model_name,
                            max_length=512, truncation=True, device=0)
        self.label_map = {'positive': 1, 'neutral': 0, 'negative': -1}

    def score_texts(self, texts, batch_size=64):
        results = []
        for i in range(0, len(texts), batch_size):
            results.extend(self.pipe(texts[i:i+batch_size]))
        return results

    def score_by_theme(self, doc_df):
        valid = doc_df[doc_df['topic_id'] != -1].copy()
        raw_scores = self.score_texts(valid['text'].tolist())
        valid['sentiment_label'] = [r['label'].lower() for r in raw_scores]
        valid['sentiment_score'] = [r['score'] * self.label_map.get(r['label'].lower(), 0) for r in raw_scores]
        summary = valid.groupby(['topic_id','topic_name']).agg(
            review_count=('text','count'),
            avg_sentiment=('sentiment_score','mean'),
            pct_positive=('sentiment_label', lambda x: (x=='positive').mean()),
            pct_negative=('sentiment_label', lambda x: (x=='negative').mean()),
        ).reset_index()
        summary['sentiment_category'] = summary['avg_sentiment'].apply(
            lambda s: 'Positive' if s > 0.2 else ('Negative' if s < -0.1 else 'Mixed'))
        return summary.sort_values('review_count', ascending=False)

Writing src/sentiment.py


In [7]:
%%writefile src/trend_scorer.py
import pandas as pd
import numpy as np
from scipy import stats

class TrendScorer:
    def __init__(self, min_periods=3):
        self.min_periods = min_periods

    def score(self, doc_df, dataset_df):
        doc_df = doc_df.copy()
        doc_df['review_date'] = dataset_df.loc[doc_df.index, 'review_date'].values
        doc_df = doc_df.dropna(subset=['review_date'])
        doc_df = doc_df[doc_df['topic_id'] != -1]
        if len(doc_df) == 0:
            return pd.DataFrame(columns=['topic_id','topic_name','slope','momentum_score','trend_label','total_reviews'])
        doc_df['month'] = doc_df['review_date'].dt.to_period('M')
        monthly = doc_df.groupby(['topic_id','topic_name','month']).size().reset_index(name='count')
        monthly['month_num'] = monthly['month'].apply(lambda p: p.ordinal)
        results = []
        for (tid, tname), grp in monthly.groupby(['topic_id','topic_name']):
            if len(grp) < self.min_periods: continue
            slope, _, r, p, _ = stats.linregress(grp['month_num'].values, grp['count'].values)
            results.append({'topic_id':tid,'topic_name':tname,'slope':slope,
                          'total_reviews':grp['count'].sum()})
        if not results:
            return pd.DataFrame(columns=['topic_id','topic_name','slope','momentum_score','trend_label','total_reviews'])
        df = pd.DataFrame(results)
        max_s = df['slope'].abs().max()
        df['momentum_score'] = df['slope'] / max_s if max_s > 0 else 0
        df['trend_label'] = df['momentum_score'].apply(
            lambda s: '↑ Rising' if s > 0.2 else ('↓ Declining' if s < -0.2 else '→ Stable'))
        return df.sort_values('momentum_score', ascending=False)

Writing src/trend_scorer.py


In [8]:
%%writefile src/narrator.py
from openai import OpenAI
import pandas as pd, json, os

def generate_insights(master_df, category='skincare'):
    client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
    top_themes = master_df.nlargest(10,'review_count')[['theme','sentiment_category','trend_label','momentum_score','review_count']].to_dict(orient='records')
    pain_points = master_df[master_df['sentiment_category'].isin(['Mixed','Negative'])][['theme','sentiment_category','avg_sentiment']].to_dict(orient='records')
    prompt = f"""Consumer insights analyst report for {category} reviews.
TOP THEMES: {json.dumps(top_themes)}
PAIN POINTS: {json.dumps(pain_points)}
Write: KEY FINDINGS, RISING OPPORTUNITIES, PAIN POINTS, STRATEGIC RECOMMENDATIONS. Be specific and data-driven."""
    response = client.chat.completions.create(
        model='gpt-4o', messages=[{'role':'user','content':prompt}], max_tokens=1000)
    return response.choices[0].message.content

Writing src/narrator.py


---
## 🔄 Cell 3 — Train Fresh OR Load Saved Model

**First time:** Run Section A (trains and saves to Drive)

**Every time after:** Run Section B (loads in ~30 seconds)

In [11]:
# ============================================================
# SECTION A — FIRST TIME ONLY (train + save)
# Comment this out after first run
# ============================================================
import sys, pandas as pd
sys.path.append('/content/src')
from preprocessor import preprocess
from topic_model import ConsumerTopicModel
from sentiment import ThemeSentimentScorer
from trend_scorer import TrendScorer
import json

# --- Load data ---
print('Loading data...')
df = pd.read_csv('/content/reviews_0-250.csv', on_bad_lines='skip', engine='python')
df = df.sample(5000, random_state=42).reset_index(drop=True)
df['review_date'] = pd.to_datetime(df['submission_time'], errors='coerce')

# --- Preprocess ---
print('Preprocessing...')
cleaned_texts, valid_idx = preprocess(df['review_text'].tolist())
print(f'Cleaned: {len(cleaned_texts)} texts')

# --- Topic model ---
print('\nFitting topic model (2-3 mins)...')
tm = ConsumerTopicModel(min_topic_size=15).build()
tm.fit(cleaned_texts)
doc_df = tm.get_topic_per_doc(cleaned_texts)
print(f'Found {len(tm.get_topic_table())} topics')

# --- Sentiment ---
print('\nScoring sentiment...')
scorer = ThemeSentimentScorer()
sentiment_df = scorer.score_by_theme(doc_df)

# --- Trend ---
print('\nScoring trends...')
trend_df = TrendScorer().score(doc_df, df)
if len(trend_df) > 0:
    master_df = sentiment_df.merge(trend_df[['topic_id','trend_label','momentum_score']], on='topic_id', how='left')
else:
    master_df = sentiment_df.copy()
    master_df['trend_label'] = '→ Stable'
    master_df['momentum_score'] = 0.0
master_df['momentum_score'] = master_df['momentum_score'].fillna(0)
master_df['trend_label'] = master_df['trend_label'].fillna('→ Stable')
master_df['theme'] = master_df['topic_name'].apply(lambda x: ' '.join(x.split('_')[1:4]).title())

# --- Save everything to Drive ---
print('\nSaving to Drive...')
DRIVE_PATH = '/drive/MyDrive/consumer-trend-predictor/models'
tm.save(f'{DRIVE_PATH}/bertopic_model')
master_df.to_parquet(f'{DRIVE_PATH}/master_results.parquet', index=False)
doc_df.to_parquet(f'{DRIVE_PATH}/doc_topics.parquet', index=False)
with open(f'{DRIVE_PATH}/cleaned_texts.json', 'w') as f:
    json.dump(cleaned_texts, f)

print('\n✅ Training complete! Everything saved to Drive.')
print('Next time, skip this cell and run Section B instead.')

Loading data...
Preprocessing...
Cleaned: 4975 texts

Fitting topic model (2-3 mins)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Scoring trends...

Saving to Drive...

✅ Training complete! Everything saved to Drive.
Next time, skip this cell and run Section B instead.


In [12]:
# ============================================================
# SECTION B — EVERY TIME AFTER FIRST RUN (load from Drive)
# Run this instead of Section A
# ============================================================
import sys, pandas as pd, json
sys.path.append('/content/src')
from topic_model import ConsumerTopicModel

DRIVE_PATH = '/drive/MyDrive/consumer-trend-predictor/models'

print('Loading saved models from Drive...')

# Load topic model
tm = ConsumerTopicModel()
tm.load(f'{DRIVE_PATH}/bertopic_model')

# Load results
master_df = pd.read_parquet(f'{DRIVE_PATH}/master_results.parquet')
doc_df = pd.read_parquet(f'{DRIVE_PATH}/doc_topics.parquet')
with open(f'{DRIVE_PATH}/cleaned_texts.json') as f:
    cleaned_texts = json.load(f)

print(f'✅ Loaded {len(master_df)} themes · {len(cleaned_texts):,} reviews')
print(master_df[['theme','sentiment_category','trend_label','momentum_score']].head(10).to_string(index=False))

Loading saved models from Drive...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Loaded 2 themes · 4,975 reviews
       theme sentiment_category trend_label  momentum_score
  It The And           Positive ↓ Declining       -1.000000
Lèvres Et Un              Mixed    → Stable       -0.008399


---
## 🚀 Cell 4 — Launch Dashboard

In [13]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px
import json, os, sys
sys.path.append('/content/src')

st.set_page_config(page_title='Consumer Trend Signal Predictor', layout='wide', page_icon='📊')
st.markdown('<style>.main{background:#0f0f0f} h1{color:#00ff88;font-family:monospace}</style>', unsafe_allow_html=True)
st.title('📊 Consumer Trend Signal Predictor')
st.caption('Upload any product review dataset → emerging themes, sentiment, trend momentum, business report')

with st.sidebar:
    st.header('⚙️ Configuration')
    openai_key = st.text_input('OpenAI API Key', type='password')
    category = st.text_input('Product Category', value='skincare')
    sample_size = st.slider('Sample Size', 1000, 10000, 5000, 500)
    run_btn = st.button('🚀 Run Analysis', use_container_width=True)

st.subheader('1. Upload Your Data')
uploaded_file = st.file_uploader('Drop any CSV with product reviews', type=['csv'])

if uploaded_file:
    df_raw = pd.read_csv(uploaded_file, on_bad_lines='skip', engine='python')
    st.success(f'✅ Loaded {len(df_raw):,} rows · {len(df_raw.columns)} columns')
    st.write('**Columns detected:**', df_raw.columns.tolist())
    col1, col2, col3 = st.columns(3)
    with col1: text_col = st.selectbox('Review Text Column', df_raw.columns)
    with col2: date_col = st.selectbox('Date Column', ['None'] + df_raw.columns.tolist())
    with col3: rating_col = st.selectbox('Rating Column', ['None'] + df_raw.columns.tolist())

    if run_btn:
        if not openai_key:
            st.error('Please enter your OpenAI API key'); st.stop()
        os.environ['OPENAI_API_KEY'] = openai_key
        df = df_raw.sample(min(sample_size, len(df_raw)), random_state=42).reset_index(drop=True)

        with st.spinner('🧹 Cleaning text...'):
            from preprocessor import preprocess
            cleaned_texts, _ = preprocess(df[text_col].tolist())
            st.success(f'✅ Preprocessed {len(cleaned_texts):,} reviews')

        with st.spinner('🔍 Discovering themes (2-3 mins)...'):
            from topic_model import ConsumerTopicModel
            tm = ConsumerTopicModel(min_topic_size=15).build()
            tm.fit(cleaned_texts)
            st.success(f'✅ Found {len(tm.get_topic_table())} consumer themes')

        with st.spinner('💬 Scoring sentiment...'):
            from sentiment import ThemeSentimentScorer
            doc_df = tm.get_topic_per_doc(cleaned_texts)
            sentiment_df = ThemeSentimentScorer().score_by_theme(doc_df)
            st.success('✅ Sentiment scored')

        with st.spinner('📈 Computing trend momentum...'):
            if date_col != 'None':
                from trend_scorer import TrendScorer
                df['review_date'] = pd.to_datetime(df[date_col], errors='coerce')
                trend_df = TrendScorer().score(doc_df, df)
                master_df = sentiment_df.merge(trend_df[['topic_id','trend_label','momentum_score']], on='topic_id', how='left') if len(trend_df) > 0 else sentiment_df.copy()
            else:
                master_df = sentiment_df.copy()
            master_df['momentum_score'] = master_df.get('momentum_score', 0.0).fillna(0)
            master_df['trend_label'] = master_df.get('trend_label', '→ Stable').fillna('→ Stable')
            master_df['theme'] = master_df['topic_name'].apply(lambda x: ' '.join(x.split('_')[1:4]).title())
            st.success('✅ Trend momentum computed')

        with st.spinner('🤖 Generating business insights...'):
            from narrator import generate_insights
            report = generate_insights(master_df, category=category)

        st.divider()
        st.subheader('2. Consumer Theme Intelligence')
        m1,m2,m3,m4 = st.columns(4)
        m1.metric('Total Themes', len(master_df))
        m2.metric('Rising Themes', len(master_df[master_df['trend_label']=='↑ Rising']))
        m3.metric('Avg Sentiment', f"{master_df['avg_sentiment'].mean():.2f}")
        m4.metric('Reviews Analyzed', f'{len(cleaned_texts):,}')

        st.dataframe(master_df[['theme','sentiment_category','trend_label','momentum_score','review_count','avg_sentiment']].sort_values('momentum_score',ascending=False).reset_index(drop=True), use_container_width=True)

        c1,c2 = st.columns(2)
        with c1:
            fig = px.bar(master_df.nlargest(15,'review_count'), x='review_count', y='theme', orientation='h',
                color='sentiment_category', color_discrete_map={'Positive':'#00ff88','Mixed':'#ffaa00','Negative':'#ff4444'}, title='Top Themes by Volume')
            fig.update_layout(paper_bgcolor='#1a1a2e', plot_bgcolor='#1a1a2e', font_color='white')
            st.plotly_chart(fig, use_container_width=True)
        with c2:
            fig2 = px.scatter(master_df, x='momentum_score', y='avg_sentiment', size='review_count',
                color='sentiment_category', hover_name='theme',
                color_discrete_map={'Positive':'#00ff88','Mixed':'#ffaa00','Negative':'#ff4444'}, title='Momentum vs Sentiment')
            fig2.update_layout(paper_bgcolor='#1a1a2e', plot_bgcolor='#1a1a2e', font_color='white')
            st.plotly_chart(fig2, use_container_width=True)

        st.divider()
        st.subheader('3. AI-Generated Business Intelligence Report')
        st.markdown(report)
        st.divider()
        st.download_button('⬇️ Download Results CSV', master_df.to_csv(index=False), file_name='consumer_trend_report.csv', mime='text/csv')

Writing app.py


In [14]:
import subprocess, time
from pyngrok import ngrok
from google.colab import userdata

# Kill old processes
!pkill -f streamlit
ngrok.kill()
time.sleep(3)

# Start Streamlit
subprocess.Popen([
    'streamlit', 'run', 'app.py',
    '--server.port=8501',
    '--server.address=0.0.0.0',
    '--server.maxUploadSize=500',
    '--server.headless=true',
    '--server.enableCORS=false',
    '--server.enableXsrfProtection=false'
])
time.sleep(8)

# Connect ngrok
ngrok.set_auth_token(userdata.get('nygrok'))
public_url = ngrok.connect(8501)
print(f'\n🚀 Dashboard live at: {public_url}')


🚀 Dashboard live at: NgrokTunnel: "https://oppressed-automatic-ultimate.ngrok-free.dev" -> "http://localhost:8501"
